# 18_02_再帰と畳み込みの一致_選択的にすると崩れること_

コード：`ch18/blocks/18_02_再帰と畳み込みの一致_選択的にすると崩れること_.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 18 章：状態空間モデル（SSM）。再帰で 1 歩ずつ計算しても、畳み込みでまとめて計算しても同じになるか
import numpy as np

rng = np.random.default_rng(0)
N, T = 4, 32                                     # 状態の次元・系列長
A = np.diag(rng.uniform(0.5, 0.95, N))           # 安定な（固有値 < 1）遷移
B, C = rng.normal(size=(N, 1)), rng.normal(size=(1, N))
x = rng.normal(size=T)

def recurrent(x, A, B, C):                       # h_t = A h_{t-1} + B x_t,  y_t = C h_t
    h, ys = np.zeros((N, 1)), []
    for xt in x:
        h = A @ h + B * xt
        ys.append((C @ h).item())
    return np.array(ys)

# 時間で変わらない SSM は、カーネル K_k = C A^k B との畳み込みと同じ（学習時に並列で計算できる）
K = np.array([(C @ np.linalg.matrix_power(A, k) @ B).item() for k in range(T)])
conv = np.array([sum(K[k] * x[t - k] for k in range(t + 1)) for t in range(T)])
print(f"時間不変：再帰と畳み込みの最大差 {np.abs(recurrent(x, A, B, C) - conv).max():.1e}")

# 選択的 SSM（Mamba の考え方）：遷移の強さを入力に応じて変える → 固定のカーネルでは書けない
def selective(x):
    h, ys = np.zeros((N, 1)), []
    for xt in x:
        forget = np.exp(-np.log1p(np.exp(xt)))   # softplus(x) が大きい入力ほど、過去を早く忘れる
        h = (A * forget) @ h + B * xt
        ys.append((C @ h).item())
    return np.array(ys)
print(f"選択的：固定カーネルの畳み込みとの最大差 {np.abs(selective(x) - conv).max():.2f}（一致しない＝走査で計算する）")

# 推論時に持つものの大きさ：KV キャッシュは系列長に比例、SSM の状態は一定
# 想定：32 層・KV ヘッド 8・ヘッド次元 128（8B 級の GQA）、SSM は状態 16 × 内部次元 4096、bf16
layers, kv_heads, d_head, state_per_layer, bytes_ = 32, 8, 128, 16 * 4096, 2
for n in (1_000, 32_000, 1_000_000):
    kv = 2 * layers * kv_heads * d_head * n * bytes_
    ssm = layers * state_per_layer * bytes_
    print(f"系列長 {n:>9,}：KV キャッシュ {kv / 2**20:10,.1f} MiB   SSM の状態 {ssm / 2**20:6.1f} MiB")
